# 05 — Fine-tune YourMT3+ on the Restrudel corpus

Goal 2 of the project: **fine-tune the released YourMT3+ checkpoint** so it
transcribes synth-heavy electronic music, using the corpus assembled in
`04_finetune_data.ipynb`. This notebook covers:

1. **Corpus composition** — what data we will fine-tune on, by category (plots).
2. **Sampling mix** — how the target (Strudel) and forgetting-mitigation
   reference sets (EGMD / MAESTRO / Slakh) are weighted in a training batch.
3. **Register a `strudel` preset** in YourMT3's `amt/src/config/data_presets.py`.
4. **Baseline to beat** — run the released checkpoint on held-out real clips.
5. **Fine-tune** — short run on Colab (1 GPU, bf16, ≪300k steps) → Drive.
6. **Evaluate** — note-level F1 on the held-out *real* electronic set vs. baseline.

> Prereq: the datasets already live in Drive under `DATA_HOME` in YourMT3's
> 16 kHz load format (built by notebook 04). This notebook reads their index
> files; it does not re-download anything.

In [ ]:
# Environment: Colab (Drive-mounted) or local checkout.
import json, os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive/restrudel")
    DATA_HOME = DRIVE / "datasets"
    if not Path("/content/restrudel").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/henrik253/Restrudel.git", "/content/restrudel"], check=True)
    REPO = Path("/content/restrudel")
else:
    REPO = Path(os.environ.get("RESTRUDEL_REPO", Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
    DATA_HOME = Path(os.environ.get("RESTRUDEL_DATA_HOME", REPO / "datasets"))

INDEX_DIR = DATA_HOME / "yourmt3_indexes"
AMT_SRC = Path(os.environ.get("RESTRUDEL_AMT_SRC", REPO / "models" / "YourMT3" / "amt" / "src"))
print(f"colab={IN_COLAB}\nrepo={REPO}\ndata_home={DATA_HOME}\nindexes={INDEX_DIR}")
assert INDEX_DIR.exists(), "no index dir — run notebook 04 first to build the datasets in Drive"

## 1. Fine-tuning corpus — composition by category

The corpus has two roles:

- **Target domain** — `strudel`: real corpus snippets + LLM-enhanced synthetic
  songs. Synth/electronic timbres — the thing the stock model fails on.
- **Forgetting-mitigation (reference)** — `egmd`, `maestro`, `slakh`: keep the
  model's general transcription ability while we specialise it.

The cell below reads every `*_file_list.json` and aggregates songs, audio hours
(`n_frames / 16000 / 3600`) and on-disk GB per dataset.

In [ ]:
# Aggregate the corpus from the index files (train+val+test only).
import subprocess

META = {  # dataset -> (label, role, domain, color)
    "strudel": ("Strudel (synth/electronic)", "target",    "electronic", "#d1495b"),
    "egmd":    ("EGMD (electronic drums)",     "reference", "electronic", "#edae49"),
    "maestro": ("MAESTRO (acoustic piano)",    "reference", "acoustic",   "#66a182"),
    "slakh":   ("Slakh (acoustic band)",       "reference", "acoustic",   "#2e4057"),
}
DIRNAME = {"slakh": "slakh2100_yourmt3_16k"}  # else f"{k}_yourmt3_16k"

def hours(fl):
    return sum(e.get("n_frames", 0) for e in fl.values()) / 16000 / 3600

agg = {k: {"train": 0, "validation": 0, "test": 0, "songs": 0, "hours": 0.0, "gb": 0.0} for k in META}
for p in sorted(INDEX_DIR.glob("*_file_list.json")):
    ds, _, split = p.stem.replace("_file_list", "").rpartition("_")
    if ds not in META or split not in ("train", "validation", "test"):
        continue
    fl = json.load(open(p))
    agg[ds][split] += len(fl)
    agg[ds]["songs"] += len(fl)
    agg[ds]["hours"] += hours(fl)
for k in META:
    d = DATA_HOME / DIRNAME.get(k, f"{k}_yourmt3_16k")
    out = subprocess.run(f"du -sb {d} 2>/dev/null", shell=True, capture_output=True, text=True).stdout
    agg[k]["gb"] = int(out.split()[0]) / 1e9 if out.strip() else 0.0

print(f"{'dataset':<10}{'role':<11}{'train':>7}{'val':>6}{'test':>6}{'songs':>8}{'hours':>8}{'GB':>7}")
print("-" * 63)
for k, (lab, role, dom, _) in META.items():
    a = agg[k]
    print(f"{k:<10}{role:<11}{a['train']:>7}{a['validation']:>6}{a['test']:>6}"
          f"{a['songs']:>8}{a['hours']:>8.1f}{a['gb']:>7.1f}")
tot = lambda m: sum(agg[k][m] for k in META)
print("-" * 63)
print(f"{'TOTAL':<21}{tot('train'):>7}{tot('validation'):>6}{tot('test'):>6}"
      f"{tot('songs'):>8}{tot('hours'):>8.1f}{tot('gb'):>7.1f}")

### 1a. Pie charts — size by dataset

Three views of the same corpus. Note how the metrics disagree: **Slakh** dominates
disk (multi-stem `.npy`), **EGMD** dominates song count and hours, while the
**target Strudel** data is tiny — which is exactly why the sampling mix (§2), not
the raw sizes, controls what the model actually sees.

In [ ]:
import matplotlib.pyplot as plt

order  = list(META)
labels = [META[k][0] for k in order]
colors = [META[k][3] for k in order]

def _autopct(vals):
    t = sum(vals)
    return lambda p: f"{p:.1f}%\n({p/100*t:,.0f})" if p > 4 else f"{p:.1f}%"

fig, axes = plt.subplots(1, 3, figsize=(16, 5.4))
for ax, (m, title) in zip(axes, [("songs", "# songs"), ("hours", "hours of audio"), ("gb", "disk size (GB)")]):
    vals = [agg[k][m] for k in order]
    ax.pie(vals, colors=colors, startangle=90, counterclock=False,
           autopct=_autopct(vals), pctdistance=0.72, textprops={"fontsize": 8},
           wedgeprops=dict(width=0.42, edgecolor="white"))
    ax.set_title(f"{title}  ·  total {sum(vals):,.0f}", fontsize=11)
fig.legend(labels, loc="lower center", ncol=4, frameon=False, fontsize=10)
fig.suptitle("Restrudel fine-tuning corpus — composition by dataset", fontsize=13, y=1.03)
plt.tight_layout(rect=[0, 0.06, 1, 1])
plt.show()

### 1b. Target vs. reference, and electronic vs. acoustic

The same hours regrouped two ways: by **training role** (what we are teaching vs.
what keeps the model from forgetting) and by **timbre domain**.

In [ ]:
from collections import defaultdict

def group(metric, keyfn):
    d = defaultdict(float)
    for k in META:
        d[keyfn(k)] += agg[k][metric]
    return d

by_role = group("hours", lambda k: "target (Strudel)" if META[k][1] == "target" else "reference (forgetting-mitigation)")
by_dom  = group("hours", lambda k: META[k][2])

fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(16, 5))
a1.pie(by_role.values(), labels=list(by_role), colors=["#d1495b", "#8d99ae"],
       autopct=lambda p: f"{p:.1f}%", startangle=90, counterclock=False,
       wedgeprops=dict(width=0.45, edgecolor="white"), textprops={"fontsize": 9})
a1.set_title("hours by role", fontsize=11)
a2.pie(by_dom.values(), labels=list(by_dom), colors=["#e07a5f", "#3d8361"],
       autopct=lambda p: f"{p:.1f}%", startangle=90, counterclock=False,
       wedgeprops=dict(width=0.45, edgecolor="white"), textprops={"fontsize": 9})
a2.set_title("hours by timbre domain", fontsize=11)

# stacked train/val/test hours per dataset (read each split's index directly)
import numpy as np
splits = ["train", "validation", "test"]
ph = {k: {s: 0.0 for s in splits} for k in order}
for k in order:
    for s in splits:
        fpath = INDEX_DIR / f"{k}_{s}_file_list.json"
        if fpath.exists():
            ph[k][s] = hours(json.load(open(fpath)))
bottom = np.zeros(len(order))
scol = {"train": "#2a9d8f", "validation": "#e9c46a", "test": "#e76f51"}
for s in splits:
    vals = [ph[k][s] for k in order]
    a3.bar([META[k][0].split(" (")[0] for k in order], vals, bottom=bottom, label=s, color=scol[s])
    bottom += vals
a3.set_ylabel("hours"); a3.set_title("train / val / test hours", fontsize=11)
a3.tick_params(axis="x", rotation=25, labelsize=8); a3.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 2. Sampling mix for fine-tuning

Fine-tuning **only** on Strudel would make the model forget everything else, so a
training batch mixes the target with the reference sets. YourMT3's original
`all_cross_final` preset weighted 10 datasets (see notebook 04 §4); here we need a
much simpler mix that **upweights the tiny target** relative to its raw size.

Below is a *starting* proposal — Strudel carries the bulk of the sampling weight
even though it is <1 % of the hours, with the reference sets sharing the rest to
hold general performance. Tune these before the real run.

In [ ]:
# Proposed sampling weights (sum to 1). NOT proportional to size — the target is upweighted.
WEIGHTS = {
    "strudel": 0.50,   # target domain — the point of the fine-tune
    "slakh":   0.25,   # richest multi-instrument reference
    "maestro": 0.15,   # pitched-note replay (piano)
    "egmd":    0.10,   # drum-kit coverage
}
assert abs(sum(WEIGHTS.values()) - 1.0) < 1e-9

raw = {k: agg[k]["hours"] for k in order}
raw_share = {k: raw[k] / sum(raw.values()) for k in order}

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 5))
a1.pie([raw_share[k] for k in order], labels=[k for k in order], colors=colors,
       autopct=lambda p: f"{p:.1f}%", startangle=90, counterclock=False,
       wedgeprops=dict(width=0.45, edgecolor="white"), textprops={"fontsize": 9})
a1.set_title("raw corpus share (by hours)", fontsize=11)
a2.pie([WEIGHTS[k] for k in order], labels=[k for k in order], colors=colors,
       autopct=lambda p: f"{p:.0f}%", startangle=90, counterclock=False,
       wedgeprops=dict(width=0.45, edgecolor="white"), textprops={"fontsize": 9})
a2.set_title("proposed sampling weight", fontsize=11)
fig.suptitle("Raw size vs. training sampling mix — the target is deliberately upweighted", y=1.02)
plt.tight_layout(); plt.show()

for k in order:
    print(f"{k:<9} raw {raw_share[k]*100:5.1f}%  ->  sampled {WEIGHTS[k]*100:4.0f}%")

## 3. Register the `strudel` presets in YourMT3

YourMT3's data loader reads named presets from `amt/src/config/data_presets.py`.
Fine-tuning needs two:

- **`strudel`** — single-source, target-only. The loader builds the file-list
  path from `dataset_name` + `*_split`, so this points at
  `strudel_{train,validation,test}_file_list.json` in `INDEX_DIR`.
- **`strudel_ft`** — the §2 fine-tuning mix: target upweighted to **0.50**, with
  slakh/maestro/egmd as forgetting-mitigation references. Structure mirrors the
  released model's own `all_*` presets.

`models/YourMT3` is fetched fresh and gitignored, so we **patch the presets file
in place at runtime** (idempotent). Fields mirror the `slakh` preset — GM
instrument + GM drum eval vocab — minus stems (our audio is a mono mix).


In [ ]:
# Patch amt/src/config/data_presets.py with `strudel` (single) + `strudel_ft` (multi).
# Idempotent: re-running is a no-op. Mirrors the `slakh` preset, minus stems.
DP = AMT_SRC / "config" / "data_presets.py"
src = DP.read_text()

SINGLE = """\
    # >>> restrudel strudel preset (injected by notebook 05) >>>
    "strudel": {
        "eval_vocab": [GM_INSTR_CLASS],
        "eval_drum_vocab": drum_vocab_presets["gm"],
        "dataset_name": "strudel",
        "train_split": "train",
        "validation_split": "validation",
        "test_split": "test",
        "has_stem": False,
    },
    # <<< restrudel strudel preset <<<
"""

MULTI = """\
    # >>> restrudel strudel_ft preset (injected by notebook 05) >>>
    "strudel_ft": {
        "presets": ["strudel", "slakh", "maestro", "egmd"],
        "weights": [0.50, 0.25, 0.15, 0.10],            # from Section 2
        "eval_vocab": [GM_INSTR_CLASS, None, None, None],
        "eval_drum_vocab": drum_vocab_presets["gm"],
        "val_max_num_files": 20,
        "test_max_num_files": None,
    },
    # <<< restrudel strudel_ft preset <<<
"""

if "restrudel strudel preset" in src:
    print("presets already patched -> skipping")
else:
    src = src.replace("data_preset_single_cfg = {\n",
                      "data_preset_single_cfg = {\n" + SINGLE, 1)
    src = src.replace("data_preset_multi_cfg = {\n",
                      "data_preset_multi_cfg = {\n" + MULTI, 1)
    DP.write_text(src)
    print("patched", DP)

# textual verify (dependency-free); GM_INSTR_CLASS resolves because slakh uses it
patched = DP.read_text()
assert '"strudel":' in patched and '"strudel_ft":' in patched, "injection failed"
print("verified: strudel + strudel_ft present in", DP.name)


### 3a. Rebase index paths onto this `DATA_HOME`

`preprocess_strudel.py` wrote **absolute build-machine paths** into the file
lists (e.g. `/Users/.../datasets/...`). On Colab the data lives under Drive, so
those paths won't resolve. Every song sits at
`<DATA_HOME>/strudel_yourmt3_16k/<strudel_id>/`, so we recompute the paths from
`strudel_id`. Idempotent; also reports any missing audio (should be 0 once the
Drive push from notebook 04 has finished).


In [ ]:
import json
SONG_ROOT = DATA_HOME / "strudel_yourmt3_16k"
for split in ("train", "validation", "test"):
    fl_path = INDEX_DIR / f"strudel_{split}_file_list.json"
    fl = json.load(open(fl_path))
    rewrote = 0
    for e in fl.values():
        sid = e.get("strudel_id")
        if not sid:
            continue
        d = SONG_ROOT / sid
        for key, fname in (("mix_audio_file", "mix.wav"),
                           ("note_events_file", f"{sid}_note_events.npy"),
                           ("notes_file",       f"{sid}_notes.npy"),
                           ("midi_file",        f"{sid}.mid")):
            if key in e and e[key] != str(d / fname):
                e[key] = str(d / fname); rewrote += 1
    json.dump(fl, open(fl_path, "w"), indent=4)
    missing = sum(1 for e in fl.values()
                  if e.get("strudel_id") and not (SONG_ROOT / e["strudel_id"] / "mix.wav").exists())
    print(f"{split:<11} {len(fl):>4} songs | rewrote {rewrote:>5} paths | mix.wav missing: {missing}")


## 4. Baseline to beat

Record what the **released** checkpoint scores untouched on the Strudel corpus
**test** split (real held-out human patterns — never seen by the generator or
the fine-tune). That note-F1 is the number to beat.

The released `YPTF.MoE+Multi (noPS)` checkpoint is rebuilt with the exact
architecture flags from `scripts/yourmt3_transcribe.py`. Full note-level F1 over
a split is what `amt/src/test.py` computes (see §6); here we just make sure the
checkpoint is present and sanity-transcribe one held-out clip end-to-end.


In [ ]:
# Ensure the released checkpoint is present (fetch once, ~1 GB), then point at a
# held-out test clip. RELEASED_EXP / RELEASED_CKPT are reused by Sections 5-6.
import subprocess, json
RELEASED_EXP  = "mc13_256_g4_all_v7_mt3f_sqr_rms_moe_wf4_n8k2_silu_rope_rp_b36_nops"
RELEASED_CKPT = AMT_SRC.parent / "logs" / "2024" / RELEASED_EXP / "checkpoints" / "last.ckpt"
if not RELEASED_CKPT.exists():
    print("fetching released model + checkpoint (~1 GB)...")
    subprocess.run([sys.executable, str(REPO / "scripts" / "fetch_yourmt3.py")], check=True)
print("baseline checkpoint present:", RELEASED_CKPT.exists())

test_fl = json.load(open(INDEX_DIR / "strudel_test_file_list.json"))
sample  = next(iter(test_fl.values()))
sample_wav = Path(sample["mix_audio_file"])
print(f"sample test clip: {sample.get('strudel_id')} -> {sample_wav}")
print("exists:", sample_wav.exists())

# Sanity transcription of ONE clip (CPU/MPS ok). Uncomment to run:
# subprocess.run([sys.executable, str(REPO / "scripts" / "yourmt3_transcribe.py"),
#                 str(sample_wav), "--out", "/tmp/baseline_pred.mid"], check=True)
# Full note-F1 over the whole split is the eval in Section 6.


## 5. Fine-tune run

Fine-tune the released checkpoint on the `strudel_ft` mix. YourMT3 has no
`--pretrained` flag (it raises `NotImplementedError`); instead we **seed a fresh
experiment dir with the released weights** — `train.py` then loads the
`state_dict` (`strict=False`) with a *fresh* optimizer/step count, i.e. real
fine-tuning, not a resume.

Two things that make this actually work:
- **Architecture flags must match the checkpoint** (from
  `scripts/yourmt3_transcribe.py`): `perceiver-tf` encoder, `multi-t5` decoder,
  MoE FF (`-nmoe 8 -kmoe 2`), RoPE, `mc13_full_plus_256` tokenizer. A mismatch
  makes `strict=False` silently drop weights → you'd be training from noise.
- **`cwd = models/YourMT3`**, because `save_dir="amt/logs"` is relative — so
  `amt/logs/ymt3/<exp>/checkpoints/` and the released
  `amt/logs/2024/<released>/...` resolve correctly.

Short Colab run: 1 GPU, `bf16-mixed`, a few epochs. Lower `-bsz` if 16 GB OOMs.
Checkpoints land in `amt/logs/ymt3/strudel_ft_v0/` — snapshot that dir to Drive.


In [ ]:
# GPU sanity check for the eventual training run.
try:
    import torch
    print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
          "|", (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU — set Runtime > Change runtime type > GPU"))
except ImportError:
    print("torch not installed in this runtime yet (only needed for §4–6)")

In [ ]:
# Seed a fresh experiment dir with the released weights and assemble the command.
import shutil
MODEL_ROOT = AMT_SRC.parents[1]          # models/YourMT3  -> cwd for train.py/test.py
AMT        = AMT_SRC.parent              # models/YourMT3/amt
FT_EXP     = "strudel_ft_v0"             # our fine-tune experiment id (project 'ymt3')
FT_CKPT    = AMT / "logs" / "ymt3" / FT_EXP / "checkpoints" / "last.ckpt"

FT_CKPT.parent.mkdir(parents=True, exist_ok=True)
if not FT_CKPT.exists():
    shutil.copy2(RELEASED_CKPT, FT_CKPT)  # copy keeps the released checkpoint pristine
print("seeded fine-tune checkpoint:", FT_CKPT.exists(), "->", FT_CKPT)

# Architecture flags — MUST match the released checkpoint (yourmt3_transcribe.py).
ARCH = ["-tk", "mc13_full_plus_256", "-dec", "multi-t5", "-nl", "26",
        "-enc", "perceiver-tf", "-sqr", "1", "-ff", "moe", "-wf", "4",
        "-nmoe", "8", "-kmoe", "2", "-act", "silu", "-epe", "rope", "-rp", "1",
        "-ac", "spec", "-hop", "300", "-atc", "1"]

# Fine-tune hyperparameters (short run; tune to your GPU).
EPOCHS, LR, BSZ = 3, "1e-4", ["-bsz", "2", "4"]     # sub/local per GPU; lower if OOM
TRAIN_CMD = [sys.executable, "amt/src/train.py", FT_EXP, "-d", "strudel_ft",
             *ARCH, "-pr", "bf16-mixed", "-e", str(EPOCHS), "-lr", LR, *BSZ, "-g", "auto"]
print("\nrun from", MODEL_ROOT, ":\n ", " ".join(TRAIN_CMD))


In [ ]:
# Launch the fine-tune (long-running; needs a GPU). train.py resolves logs
# relative to cwd, so we run it from MODEL_ROOT.
assert __import__("torch").cuda.is_available(), "no GPU — set Runtime > Change runtime type > GPU"
proc = subprocess.run(TRAIN_CMD, cwd=MODEL_ROOT)
print("train.py exit:", proc.returncode)
# Snapshot the run to Drive so it survives the Colab session:
# shutil.copytree(AMT / "logs" / "ymt3" / FT_EXP, DRIVE / "checkpoints" / FT_EXP, dirs_exist_ok=True)


## 6. Evaluate vs. baseline

Note-level F1 on the held-out Strudel **test** split (their own `test.py` loop),
for the **baseline** (released checkpoint) vs. the **fine-tune**. A win here — the
fine-tune beating its own baseline on synth timbres — is the thesis result
(Milestone M6). Extend the same eval to real electronic clips in `mp3s/`.

Both runs use the `strudel` (target-only) preset and the §5 `ARCH` flags; only
the experiment id / project differ. Needs a GPU for the full split.


In [ ]:
# Build the eval commands for baseline vs. fine-tune (run from MODEL_ROOT).
# Requires ARCH / MODEL_ROOT / FT_EXP / RELEASED_EXP from Sections 4-5.
def eval_cmd(exp_id, project):
    return [sys.executable, "amt/src/test.py", exp_id, "-p", project,
            "-d", "strudel", *ARCH, "-pr", "32", "-g", "auto"]

print("baseline :  cd", MODEL_ROOT, "&&", " ".join(eval_cmd(RELEASED_EXP, "2024")))
print("finetuned:  cd", MODEL_ROOT, "&&", " ".join(eval_cmd(FT_EXP, "ymt3")))
# subprocess.run(eval_cmd(RELEASED_EXP, "2024"), cwd=MODEL_ROOT)   # baseline F1
# subprocess.run(eval_cmd(FT_EXP, "ymt3"),       cwd=MODEL_ROOT)   # fine-tuned F1
